## class AppConfig

> **Note**
- Khi chúng ta tiến đến Chặng 7 (đọc file huấn luyện của tác giả), nếu thầy trò ta phát hiện ra tác giả dùng những con số khác, ta chỉ cần quay lên đầu Notebook, sửa một con số trong class Config là toàn bộ dự án sẽ tự động cập nhật theo.

In [ ]:
import os
import random
import numpy as np
import torch

In [ ]:
from dataclasses import dataclass

# Cho Logger
class SystemConfig:
    """Cấu hình vận hành hệ thống."""
    log_file_path: str = '/content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/logs/pipeline.log'

# Cho tiền xử lý tĩnh
@dataclass(frozen=True) # Không được sửa trong chương trình
class DataConfig:
    # Đường dẫn file mapping
    TACO_TO_7_CLASSES_MAP: str = '/content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/config/mapping_label.json'
    RAW_ANNOTATIONS_PATH: str = '/content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/data/raw/annotations.json'

    # Đường dẫn lưu processed anns
    PATH_7_CLASSES: str = '/content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/data/processed/taco_to_detectwaste_annotations.json'

    # Đường dẫn lưu train/test
    PATH_MULTI_TRAIN: str = '/content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/data/train/multi_train_annotations.json'
    PATH_MULTI_TEST: str = '/content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/data/test/multi_test_annotations.json'

    PATH_BINARY_TRAIN: str = '/content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/data/train/binary_train_annotations.json'
    PATH_BINARY_TEST: str = '/content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/data/test/binary_test_annotations.json'

# Tổng hợp
@dataclass
class AppConfig:
    system: SystemConfig = SystemConfig()
    data: DataConfig = DataConfig()

# Utils

## class LoggerSetup

In [ ]:
import os
import logging
import sys
import atexit

class LoggerSetup:
    """
    Trạm điều phối Cấu hình Log tập trung cho toàn bộ dự án.
    Thiết kế chuẩn MLOps: Tự động khởi tạo và tự động đóng luồng ghi.
    """

    @staticmethod
    def initialize(log_file: str, clear_old_logs: bool = True):
        # 1. Tạo thư mục cha
        log_dir = os.path.dirname(log_file)
        if log_dir:
            os.makedirs(log_dir, exist_ok=True)

        # 2. Làm sạch cấu hình cũ (Tránh duplicate log trên Colab)
        root = logging.getLogger()
        root.handlers = []

        # 3. Cấu hình Format & Mode
        formatter = logging.Formatter('%(asctime)s - [%(name)s] - %(levelname)s - %(message)s')
        file_mode = 'w' if clear_old_logs else 'a'

        # TẦNG 1: FILE HANDLER (DEBUG)
        # delay=False: Ép tạo file ngay lập tức
        fh = logging.FileHandler(log_file, mode=file_mode, encoding='utf-8', delay=False)
        fh.setLevel(logging.DEBUG)
        fh.setFormatter(formatter)
        root.addHandler(fh)

        # TẦNG 2: STREAM HANDLER (INFO)
        ch = logging.StreamHandler(sys.stdout)
        ch.setLevel(logging.INFO)
        ch.setFormatter(formatter)
        root.addHandler(ch)

        root.setLevel(logging.DEBUG)

        # Đăng ký tự động đóng log khi kết thúc chương trình
        def shutdown_logging():
            fh.flush()
            fh.close()
            root.removeHandler(fh)
            root.removeHandler(ch)

        atexit.register(shutdown_logging)

        logging.info(f"Hệ thống Logger đã sẵn sàng. File log: {log_file}")

## class DatasetUtils

Chuyên xử lý các mảng dữ liệu (Array), từ điển (Dictionary), lọc dữ liệu (Filter), tính toán Bảng băm (Hash Set). Nếu cấu trúc JSON bị sai key, DatasetUtils sẽ báo lỗi trên RAM

In [ ]:
import logging
from typing import List, Dict, Any
# Giả định bạn đã đặt class LoggerSetup ở file logger_setup.py
# from utils.logger_setup import LoggerSetup

# Khởi tạo Lính gác (Logger) riêng cho module Utils này
logger = logging.getLogger("DatasetUtils")

class DatasetUtils:
    """
    Tập hợp các hàm Tiện ích (Stateless) chuyên xử lý dữ liệu JSON/List.
    """

    @staticmethod
    def filter_annotations(annotations: List[Dict[str, Any]], images: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Trích xuất các nhãn tương ứng với danh sách ảnh đầu vào.
        Tối ưu tốc độ O(1) bằng Hash Set. Tích hợp Logging giám sát luồng dữ liệu.
        """
        logger.info(f"Bắt đầu lọc nhãn cho {len(images)} bức ảnh...")

        # 1. Tạo Hash Set chứa ID ảnh hợp lệ
        valid_image_ids = {int(im['id']) for im in images}

        # 2. Lọc nhãn bằng Bảng băm
        filtered_anns = [
            ann for ann in annotations
            if int(ann['image_id']) in valid_image_ids
        ]

        # 3. Báo cáo tỷ lệ hao hụt (Rất quan trọng để Debug Pipeline)
        drop_rate = len(annotations) - len(filtered_anns)
        logger.info(f"Hoàn tất lọc: Giữ lại {len(filtered_anns)} / {len(annotations)} nhãn. Đã loại bỏ {drop_rate} nhãn rác.")

        return filtered_anns

    @staticmethod
    def concatenate_datasets(list_of_datasets: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Dung hợp (Concatenate) nhiều COCO Datasets (đã nạp trên RAM) thành một Siêu tập dữ liệu.
        Xử lý xung đột ID bằng cách cấp phát lại ID tuyến tính.
        """
        if not list_of_datasets:
            logger.warning("Danh sách dataset đầu vào rỗng. Trả về dictionary trống.")
            return {}

        logger.info(f"Tiến hành gộp {len(list_of_datasets)} bộ dữ liệu...")

        # Khởi tạo Kho Tổng (Bắt đầu ID từ 1 theo chuẩn COCO)
        last_im_id = 1
        last_ann_id = 1

        concat_dataset = {
            'info': {},
            'licenses': [],
            'categories': [],
            'images': [],
            'annotations': []
        }

        for index, dataset in enumerate(list_of_datasets):
            # Lấy Metadata & Categories của bộ dữ liệu ĐẦU TIÊN làm gốc (Anchor)
            if index == 0:
                concat_dataset['info'] = dataset.get('info', {})
                concat_dataset['licenses'] = dataset.get('licenses', [])
                concat_dataset['categories'] = dataset.get('categories', [])

            # Sổ tay lưu ánh xạ: ID Ảnh Cũ -> ID Ảnh Mới cho từng dataset
            img_id_mapping = {}

            # 1. Cập nhật Primary Key cho Ảnh
            for im in dataset.get('images', []):
                img_id_mapping[im['id']] = last_im_id
                im['id'] = last_im_id
                last_im_id += 1

            # 2. Cập nhật Foreign Key cho Nhãn
            for ann in dataset.get('annotations', []):
                # An toàn với .get() nếu nhãn trỏ về ảnh không tồn tại
                ann['image_id'] = img_id_mapping.get(ann['image_id'], ann['image_id'])
                ann['id'] = last_ann_id
                last_ann_id += 1

            # 3. Đổ dữ liệu vào Kho Tổng bằng .extend() tối ưu RAM
            concat_dataset['images'].extend(dataset.get('images', []))
            concat_dataset['annotations'].extend(dataset.get('annotations', []))

        logger.info(f"Gộp thành công! Tổng số ảnh: {len(concat_dataset['images'])}, Tổng số nhãn: {len(concat_dataset['annotations'])}.")

        return concat_dataset

In [ ]:
import logging
from typing import List, Dict, Any

logger = logging.getLogger("DatasetUtils")

class DatasetUtils:
    """
    Tập hợp các hàm Tiện ích (Stateless) chuyên xử lý dữ liệu JSON/List.
    """

    @staticmethod
    def filter_annotations(annotations: List[Dict[str, Any]], images: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Trích xuất các nhãn tương ứng với danh sách ảnh đầu vào.
        Tối ưu tốc độ O(1) bằng Hash Set. Tích hợp Logging giám sát luồng dữ liệu.
        """
        logger.info(f"Bắt đầu lọc nhãn cho {len(images)} bức ảnh...")

        # 1. Tạo Hash Set chứa ID ảnh hợp lệ
        valid_image_ids = {int(im['id']) for im in images}
        logger.debug(f"DatasetUtils: Đã tạo Hash Set với {len(valid_image_ids)} Image IDs.")

        # 2. Lọc nhãn bằng Bảng băm
        filtered_anns = [
            ann for ann in annotations
            if int(ann['image_id']) in valid_image_ids
        ]

        # 3. Báo cáo tỷ lệ hao hụt
        drop_rate = len(annotations) - len(filtered_anns)
        logger.info(f"Hoàn tất lọc: Giữ lại {len(filtered_anns)} / {len(annotations)} nhãn. Đã loại bỏ {drop_rate} nhãn không thuộc danh sách ảnh.")
        return filtered_anns

    @staticmethod
    def concatenate_datasets(list_of_datasets: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Dung hợp (Concatenate) nhiều COCO Datasets (đã nạp trên RAM) thành một Siêu tập dữ liệu.
        Xử lý xung đột ID bằng cách cấp phát lại ID tuyến tính.
        """
        if not list_of_datasets:
            logger.warning("DatasetUtils: Danh sách dataset rỗng. Trả về dictionary trống.")
            return {}

        logger.info(f"Tiến hành gộp {len(list_of_datasets)} bộ dữ liệu...")

        # Khởi tạo Kho Tổng (Bắt đầu ID từ 1 theo chuẩn COCO)
        last_im_id = 1
        last_ann_id = 1

        concat_dataset = {
            'info': {},
            'licenses': [],
            'categories': [],
            'images': [],
            'annotations': []
        }

        for index, dataset in enumerate(list_of_datasets):
            logger.debug(f"DatasetUtils: Đang xử lý dataset thứ {index} | Current Image ID start: {last_im_id}")
            if index == 0:
                concat_dataset['info'] = dataset.get('info', {})
                concat_dataset['categories'] = dataset.get('categories', [])

            img_id_mapping = {}
            for im in dataset.get('images', []):
                old_id = im['id']
                img_id_mapping[old_id] = last_im_id
                im['id'] = last_im_id
                logger.debug(f"DatasetUtils: Mapping Image ID {old_id} -> {last_im_id}")
                last_im_id += 1

            for ann in dataset.get('annotations', []):
                ann['image_id'] = img_id_mapping.get(ann['image_id'], ann['image_id'])
                ann['id'] = last_ann_id
                logger.debug(f"DatasetUtils: Mapping Annotation ID {ann['id']} to Image ID {ann['image_id']}")
                last_ann_id += 1

            concat_dataset['images'].extend(dataset.get('images', []))
            concat_dataset['annotations'].extend(dataset.get('annotations', []))

        logger.info(f"Gộp thành công! Kết quả: {len(concat_dataset['images'])} ảnh, {len(concat_dataset['annotations'])} nhãn.")
        return concat_dataset

## class IOUtils

Chuyên tương tác với hệ điều hành: Mở file, Ghi file, Đọc luồng stream, Tạo thư mục. Nếu ổ cứng bị đầy, hoặc file bị khóa quyền truy cập (Permission Denied), IOUtils sẽ báo lỗi.

In [ ]:
import os
import json
import logging
from typing import Dict, List, Any

# Kích hoạt lính gác Logger cho riêng module IO
logger =  logging.getLogger("IOUtils")

class IOUtils:
    """
    Hộp chứa công cụ chuyên biệt cho các thao tác Input/Output (Đọc/Ghi file, Quản lý thư mục).
    """

    @staticmethod
    def save_coco_json(dest_path: str, dataset: Dict[str, Any]) -> Dict[str, Any]:
        """
        Lưu trữ Sổ cái COCO xuống ổ đĩa.
        Bảo vệ cấu trúc: Tự động bổ sung các trường bắt buộc của COCO nếu bị khuyết.
        """
        logger.info(f"Chuẩn bị đóng gói dữ liệu COCO và lưu tại: {dest_path}")

        # 1. Định hình cấu trúc xương sống (Bảo vệ an toàn cấu trúc COCO)
        data_dict = {
            'info': dataset.get('info', {}),
            'licenses': dataset.get('licenses', []),
            'images': dataset.get('images', []),
            'annotations': dataset.get('annotations', []),
            'categories': dataset.get('categories', [])
        }

        # 2. Xả dữ liệu xuống ổ cứng an toàn (Safe I/O)
        try:
            # Tự động tạo thư mục cha nếu nó chưa tồn tại (chống lỗi FileNotFoundError)
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            logger.debug(f"IOUtils: Đảm bảo thư mục {os.path.dirname(dest_path)} đã sẵn sàng.")

            with open(dest_path, 'w', encoding='utf-8') as f:
                json.dump(data_dict, f, indent=2, sort_keys=True)

            logger.info(f"Lưu file thành công! Lưu {dest_path}, Kích thước: {len(data_dict['images'])} ảnh, {len(data_dict['annotations'])} nhãn.")

        except Exception as e:
            # Bắt lỗi ổ cứng (VD: hết dung lượng, sai quyền truy cập)
            logger.exception(f"[SYSTEM-ERROR] Thất bại khi ghi file xuống ổ cứng ({dest_path}): {e}")
            raise e # Ném lỗi lên trên để Pipeline tổng biết mà dừng băng chuyền

        return data_dict

    @staticmethod
    def load_json(source_path: str) -> Dict[str, Any]:
        """
        Nạp dữ liệu JSON từ ổ cứng lên RAM một cách an toàn.
        Tối ưu hóa: Dùng cơ chế Stream (json.load) để chống tràn bộ nhớ (OOM).
        """
        logger.info(f"Đang nạp dữ liệu từ file: {source_path}...")

        # 1. Bảo vệ Hệ thống: Kiểm tra file có tồn tại không trước khi đọc
        if not os.path.exists(source_path):
            logger.error(f"[SYSTEM-ERROR] Không tìm thấy file tại đường dẫn: {source_path}")
            raise FileNotFoundError(f"Lỗi: File {source_path} không tồn tại!")

        # 2. Xử lý Đọc Luồng (Safe Stream I/O)
        try:
            logger.debug(f"IOUtils: Bắt đầu đọc stream JSON từ {source_path}")
            # Dùng json.load() thay vì json.loads(f.read())
            with open(source_path, 'r', encoding='utf-8') as f:
                data_dict = json.load(f)

            # (Tùy chọn) Tính toán kích thước file để log báo cáo
            file_size_mb = os.path.getsize(source_path) / (1024 * 1024)
            logger.info(f"Nạp thành công {source_path}! Dung lượng file: {file_size_mb:.2f} MB")

            return data_dict

        except json.JSONDecodeError as e:
            # Bắt lỗi file JSON bị hỏng (corrupted)
            logger.exception(f"[SYSTEM-ERROR] File JSON bị lỗi cấu trúc ({source_path}). Chi tiết: {e}")
            raise e
        except Exception as e:
            # Bắt các lỗi I/O khác (ví dụ: đang đọc thì ổ cứng bị rút ra)
            logger.exception(f"[SYSTEM-ERROR] Lỗi I/O không xác định khi đọc file: {e}")
            raise e

# Preprocessing Classes

Các class xử lý json thô trên RAM

## class BaseAnnotationProcessor(ABC)

In [ ]:
from abc import ABC, abstractmethod
import logging

logger = logging.getLogger("BaseProcessor")

class BaseAnnotationProcessor(ABC):
    """
    Lớp Trừu tượng cốt lõi cho việc biến đổi dữ liệu JSON trên RAM.
    """

    def __init__(self, dataset_dict: dict):
        self.dataset = dataset_dict
        logger.info(f"[{self.__class__.__name__}] Đã khởi tạo processor với tập dữ liệu chứa {len(self.dataset.get('images', []))} ảnh.")

    def get_dataset(self) -> dict:
        return self.dataset

    @abstractmethod
    def transform(self):
        pass

In [ ]:
import json
import logging

logger = logging.getLogger("CategoryMappingProcessor")

class CategoryMappingProcessor(BaseAnnotationProcessor):
    def __init__(self, dataset_dict: dict, mapping_dict: dict):
        super().__init__(dataset_dict)
        logger.info("CategoryMappingProcessor: Đang khởi tạo bảng tra cứu ánh xạ (Lookup Table)...")
        self.fast_lookup_dict = self._build_fast_lookup(mapping_dict)
        logger.info(f"CategoryMappingProcessor: Đã nạp {len(self.fast_lookup_dict)} quy tắc ánh xạ nhãn.")

    def _build_fast_lookup(self, config: dict) -> dict:
        lookup = {}
        for new_label, old_labels_list in config.items():
            for old_label in old_labels_list:
                lookup[old_label] = new_label
                logger.debug(f"CategoryMappingProcessor: Map '{old_label}' -> '{new_label}'")
        return lookup

    def transform(self):
        logger.info("CategoryMappingProcessor: Bắt đầu quá trình đồng bộ hóa danh mục...")

        new_categories = []
        new_group_to_new_id = {}
        old_id_to_new_id = {}
        current_new_id = 1

        for old_cat in self.dataset.get('categories', []):
            old_name = old_cat['name']
            new_name = self.fast_lookup_dict.get(old_name, 'unknown')

            if new_name not in new_group_to_new_id:
                new_group_to_new_id[new_name] = current_new_id
                new_categories.append({"id": current_new_id, "name": new_name, "supercategory": new_name})
                logger.debug(f"CategoryMappingProcessor: Tạo danh mục mới '{new_name}' with ID {current_new_id}")
                current_new_id += 1

            old_id_to_new_id[old_cat['id']] = new_group_to_new_id[new_name]
            logger.debug(f"CategoryMappingProcessor: Category Mapping ID {old_cat['id']} -> {old_id_to_new_id[old_cat['id']]}")

        old_count = len(self.dataset.get('categories', []))
        self.dataset['categories'] = new_categories

        logger.info(f"CategoryMappingProcessor: Cập nhật Annotations... (Đang xử lý {len(self.dataset.get('annotations', []))} nhãn)")
        for ann in self.dataset.get('annotations', []):
            old_cat_id = ann['category_id']
            ann['category_id'] = old_id_to_new_id.get(old_cat_id, old_cat_id)
            ann.pop('segmentation', None)

        logger.info(f"CategoryMappingProcessor: Hoàn tất! Thu gọn {old_count} loại nhãn gốc về {len(new_categories)} loại nhãn.")

In [ ]:
# from core.processors.base_processor import BaseAnnotationProcessor
# from utils.logger_setup import LoggerSetup

# Kích hoạt lính gác Logger
logger =  logging.getLogger("BinarizationProcessor")

class BinarizationProcessor(BaseAnnotationProcessor):
    """
    Cỗ máy San phẳng Nhãn dán (Binarization).
    Biến đổi mọi phân loại rác (Multi-class) về một nhãn duy nhất (Binary/Litter).
    Hoạt động hoàn toàn trên RAM, không phụ thuộc ổ cứng.
    """

    def __init__(self, dataset_dict: dict):
        # Truyền "Sổ cái" cho lớp Cha quản lý
        super().__init__(dataset_dict)

        # Định nghĩa hằng số cho chuẩn Binary
        self.binary_category_id = 1
        self.binary_name = 'litter'

    def transform(self):
        """Thực thi biến đổi dữ liệu lõi trên RAM."""
        logger.info("Bắt đầu tiến trình San phẳng Nhãn (Binarization)...")

        # 1. Cập nhật Meta-data
        # Khởi tạo dict 'info' nếu file JSON gốc bị khuyết
        if 'info' not in self.dataset:
            self.dataset['info'] = {}

        self.dataset['info']['description'] = 'detectwaste_binary'
        self.dataset['info']['year'] = 2021

        # 2. Xây dựng Menu chuẩn mới (Chỉ có đúng 1 danh mục)
        # Ứng dụng bài học trước: Giữ supercategory và cho nó bằng chính 'litter'
        len_old_cat = len(self.dataset['categories'])
        self.dataset['categories'] = [{
            'id': self.binary_category_id,
            'name': self.binary_name,
            'supercategory': self.binary_name
        }]

        # 3. Đồng bộ lại toàn bộ Hóa đơn (Annotations)
        annotations = self.dataset.get('annotations', [])

        for ann in annotations:
            # Lấy "bút xóa" bôi mã cũ, ghi mã số 1 vào
            ann['category_id'] = self.binary_category_id

            # Xóa trường segmentation làm nhẹ file (nhất quán với các processor khác)
            ann.pop('segmentation', None)

        logger.info(f"Hoàn tất! {len_old_cat} loại nhãn đã được quy về ID {self.binary_category_id} ({self.binary_name}).")

## class BaseDatasetSplitter(ABC)

In [ ]:
from abc import ABC, abstractmethod
from typing import Dict, Any, Tuple, List
# Gọi chuyên gia xử lý mảng mà chúng ta đã viết ở phần Utils
# from utils.dataset_utils import DatasetUtils
# from utils.logger_setup import LoggerSetup

logger =  logging.getLogger("BaseDatasetSplitter")

class BaseDatasetSplitter(ABC):
    """
    Lớp Trừu tượng cốt lõi cho mọi Động cơ chia rổ dữ liệu (Train/Test Split).
    Áp dụng Design Pattern: Template Method.
    """

    def __init__(self, test_size: float = 0.2, random_state: int = 2020):
        # Lưu các siêu tham số dùng chung cho mọi thuật toán chia
        self.test_size = test_size
        self.random_state = random_state

    @abstractmethod
    def split(self, dataset: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        """
        [BẮT BUỘC]: Bản hợp đồng yêu cầu mọi Class con phải tự viết thuật toán chia.
        Input: Sổ cái COCO JSON (Toàn bộ dữ liệu).
        Output: Trả về một Tuple chứa 2 Sổ cái mới (Train Dataset, Test Dataset).
        """
        pass

    def _build_coco_subset(self, original_dataset: Dict[str, Any], subset_images: List[Dict]) -> Dict[str, Any]:
        """
        [HÀM TIỆN ÍCH DÙNG CHUNG CỦA LỚP CHA]
        Đóng gói một danh sách ảnh rời rạc trở thành một Sổ cái COCO hoàn chỉnh.
        Các class con chỉ cần gọi hàm này sau khi chia ảnh xong.
        """
        # 1. Gọi Lính đánh thuê (DatasetUtils) để lọc nhãn khớp với ảnh
        subset_annotations = DatasetUtils.filter_annotations(
            annotations=original_dataset.get('annotations', []),
            images=subset_images
        )

        # 2. Xây dựng Sổ cái mới (Giữ nguyên Meta-data, chỉ thay đổi Ảnh và Nhãn)
        subset_dataset = {
            'info': original_dataset.get('info', {}),
            'licenses': original_dataset.get('licenses', []),
            'categories': original_dataset.get('categories', []),
            'images': subset_images,
            'annotations': subset_annotations
        }

        return subset_dataset

In [ ]:
!pip install iterative-stratification

In [ ]:
import numpy as np
from collections import defaultdict, Counter
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from typing import Dict, Any, Tuple, List
import logging

logger = logging.getLogger("MultiLabelSplitter")

class MultiLabelSplitter(BaseDatasetSplitter):
    def _build_dense_feature_matrix(self, images: List[Dict], annotations: List[Dict]) -> np.ndarray:
        logger.info(f"MultiLabelSplitter: Đang xây dựng ma trận đặc trưng cho {len(images)} ảnh...")
        cat_to_idx = {cat['id']: i for i, cat in enumerate(self.categories)}
        matrix = np.zeros((len(images), len(self.categories)))

        img_id_to_idx = {img['id']: i for i, img in enumerate(images)}

        for ann in annotations:
            if ann['image_id'] in img_id_to_idx:
                row = img_id_to_idx[ann['image_id']]
                col = cat_to_idx[ann['category_id']]
                matrix[row, col] += 1
                logger.debug(f"MultiLabelSplitter: Image {ann['image_id']} increment Category {ann['category_id']}")

        return matrix

    def split(self, dataset: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        images = dataset.get('images', [])
        annotations = dataset.get('annotations', [])
        self.categories = dataset.get('categories', [])

        logger.info(f"MultiLabelSplitter: Bắt đầu tiến trình chia tách {len(images)} ảnh bằng Multilabel Stratified Shuffle Split.")

        feature_matrix = self._build_dense_feature_matrix(images, annotations)
        logger.debug(f"MultiLabelSplitter: Ma trận đặc trưng đã sẵn sàng: {feature_matrix.shape}")

        strat_split = MultilabelStratifiedShuffleSplit(
            n_splits=1, test_size=self.test_size, random_state=self.random_state
        )

        logger.info(f"MultiLabelSplitter: Đang tính toán phân bổ tầng (Stratification)...")
        train_index, test_index = next(strat_split.split(images, feature_matrix))
        logger.debug(f"MultiLabelSplitter: Split hoàn tất. Train Indices count: {len(train_index)}, Test Indices count: {len(test_index)}")

        train_images = [images[i] for i in train_index]
        test_images = [images[i] for i in test_index]

        logger.info(f"MultiLabelSplitter: Kết quả chia: Train={len(train_images)} ảnh, Test={len(test_images)} ảnh.")

        train_dataset = self._build_coco_subset(dataset, train_images)
        test_dataset = self._build_coco_subset(dataset, test_images)

        return train_dataset, test_dataset

In [ ]:
import numpy as np
from collections import defaultdict, Counter
from sklearn.model_selection import StratifiedShuffleSplit
from typing import Dict, Any, Tuple, List
import logging # chỉ cần gọi thư viện logging gốc của Python, nó sẽ tự động thừa kế mọi cấu hình mà em đã set ở main.py

# from core.splitters.base_splitter import BaseDatasetSplitter

logger = logging.getLogger("SingleLabelSplitter")

class SingleLabelSplitter(BaseDatasetSplitter):
    """
    Cỗ máy chia rổ Phân tầng Giả (Pseudo-Stratified Split).
    Đại diện cho mỗi bức ảnh bằng 'Nhãn thống trị' (xuất hiện nhiều nhất)
    để ép bài toán Đa nhãn về chuẩn Đơn nhãn của scikit-learn.
    """

    def _build_dominant_label_array(self, images: List[Dict], annotations: List[Dict]) -> np.ndarray:
        """
        [Hàm nội bộ]: Thống kê và trích xuất mảng Nhãn thống trị.
        Bảo vệ hệ thống khỏi lỗi lệch pha (Misalignment) và ảnh rỗng.
        """
        logger.info("Đang trích xuất Mảng Nhãn thống trị (Dominant Label)...")
        categories_per_image = defaultdict(Counter)

        # 1. Thống kê số lượng rác
        for ann in annotations:
            categories_per_image[ann['image_id']][ann['category_id']] += 1

        # 2. Rút trích nhãn đại diện (Quét theo đúng thứ tự mảng images)
        max_category = []
        for im in images:
            im_id = im['id']
            if im_id in categories_per_image:
                # Lấy ID của loại rác xuất hiện nhiều nhất
                dominant_cat = categories_per_image[im_id].most_common(1)[0][0]
                max_category.append(dominant_cat)
            else:
                # Ảnh không có rác: Gán cờ 0 (Background) để chia rổ cho đều
                max_category.append(0)

        return np.array(max_category)

    def split(self, dataset: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        """
        Thực thi chia rổ Sổ cái COCO thành 2 tập Train/Test riêng biệt.
        """
        images = dataset.get('images', [])
        annotations = dataset.get('annotations', [])

        if not images:
            logger.error("[SYSTEM-ERROR] Dataset rỗng!")
            raise ValueError("Không thể chia rổ một Dataset rỗng.")

        # 1. Tính toán Mảng Nhãn đại diện
        dominant_labels = self._build_dominant_label_array(images, annotations)

        # 2. Kích hoạt cỗ máy StratifiedShuffleSplit (Cố định n_splits = 1)
        logger.info(f"Kích hoạt phân tầng đơn nhãn (test_size={self.test_size}, seed={self.random_state})...")
        strat_split = StratifiedShuffleSplit(
            n_splits=1,
            test_size=self.test_size,
            random_state=self.random_state
        )

        # 3. Trích xuất Index và cắt rổ ảnh
        # Dùng next() để lấy ngay kết quả của vòng lặp duy nhất
        train_index, test_index = next(strat_split.split(images, dominant_labels))

        train_images = [images[i] for i in train_index]
        test_images = [images[i] for i in test_index]

        logger.info(f"Cắt rổ thành công! Tập Train: {len(train_images)} ảnh | Tập Test: {len(test_images)} ảnh.")

        # 4. Giao quyền đóng gói COCO JSON cho Lớp Cha
        train_dataset = self._build_coco_subset(dataset, train_images)
        test_dataset = self._build_coco_subset(dataset, test_images)

        return train_dataset, test_dataset

## class BasePipeline(ABC)

In [ ]:
from abc import ABC, abstractmethod

class BasePipeline(ABC):
    """
    Lớp Trừu tượng cốt lõi cho mọi Dây chuyền Dữ liệu (Automated Assembly Lines).
    """

    @abstractmethod
    def execute(self):
        """
        [BẮT BUỘC]: Khởi động toàn bộ quy trình từ đầu đến cuối.
        """
        pass

In [ ]:
import logging
logger = logging.getLogger("TACO_Pipeline")

class WastePreprocessingPipeline(BasePipeline):
    def __init__(self, config):
        self.config = config
        logger.info("WastePreprocessingPipeline: Dây chuyền đã được cấu hình và sẵn sàng kích hoạt.")

    def execute(self):
        logger.info("--- GIAI ĐOẠN TIỀN XỬ LÝ TĨNH (XỬ LÝ RAW ANNS) ---")

        # Bước 1
        logger.info("[Pipeline-Step 1/4] Đang nạp tài nguyên từ ổ cứng...")
        raw_dataset = IOUtils.load_json(self.config.RAW_ANNOTATIONS_PATH)
        mapping_dict = IOUtils.load_json(self.config.TACO_TO_7_CLASSES_MAP)
        logger.debug("Pipeline: Tài nguyên đầu vào đã được nạp lên RAM.")

        # Bước 2
        logger.info("[Pipeline-Step 2/4] Đang tiến hành chuẩn hóa nhãn (Standardization)... CategoryMapping")
        mapper = CategoryMappingProcessor(dataset_dict=raw_dataset, mapping_dict=mapping_dict)
        mapper.transform()
        dataset_7_classes = mapper.get_dataset()
        IOUtils.save_coco_json(self.config.PATH_7_CLASSES, dataset_7_classes)
        logger.debug("Pipeline: Hoàn tất chuẩn hóa 7 danh mục.")

        # Bước 3
        logger.info("[Pipeline-Step 3/4] Đang thực hiện chia tách dữ liệu Stratified...")
        splitter = MultiLabelSplitter(test_size=0.2, random_state=2020)
        train_7_classes, test_7_classes = splitter.split(dataset_7_classes)
        IOUtils.save_coco_json(self.config.PATH_MULTI_TRAIN, train_7_classes)
        IOUtils.save_coco_json(self.config.PATH_MULTI_TEST, test_7_classes)
        logger.debug("Pipeline: Hoàn tất chia tách Train/Test.")

        # Bước 4
        logger.info("[Pipeline-Step 4/4] Đang thực hiện Binarization cho các tập dữ liệu...")
        for tag, ds, path in [('TRAIN', train_7_classes, self.config.PATH_BINARY_TRAIN), ('TEST', test_7_classes, self.config.PATH_BINARY_TEST)]:
            logger.info(f"   -> Đang Binarize tập {tag}...")
            processor = BinarizationProcessor(dataset_dict=ds)
            processor.transform()
            IOUtils.save_coco_json(path, processor.get_dataset())
            logger.debug(f"Pipeline: Hoàn tất Binarize cho tập {tag}.")

        logger.info("--- PIPELINE KẾT THÚC THÀNH CÔNG ---")

# Tải ảnh và xử lý ảnh

## class BaseTransform(ABC)
Định nghĩa khung chuẩn cho phép biến đổi.

In [ ]:
from abc import ABC, abstractmethod
from typing import Dict, Any, Tuple
import numpy as np

class BaseTransform(ABC):
    """
    Lớp Trừu tượng (Abstract Base Class) cốt lõi cho mọi thao tác biến đổi dữ liệu.
    Sử dụng Dunder method __call__ để Class hoạt động như một hàm (Callable).
    """
    @abstractmethod
    def __call__(self, image: np.ndarray, target: Dict[str, Any]) -> Tuple[np.ndarray, Dict[str, Any]]:
      pass

In [ ]:
import torch
import cv2
import numpy as np
import random

# =====================================================================
# 1. BASIC TRANSFORMS (TIỀN XỬ LÝ TĨNH)
# Dùng để ép kiểu dữ liệu và kích thước ảnh cho mô hình
# =====================================================================

class Compose(BaseTransform):
    """Đóng gói nhiều phép biến đổi thành một dây chuyền (Pipeline)."""
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target

class ToTensor(BaseTransform):
    """Chuyển đổi ma trận ảnh numpy (H, W, C) sang chuẩn PyTorch Tensor (C, H, W)."""
    def __call__(self, image, target):
        image = torch.from_numpy(image.transpose((2, 0, 1))).contiguous()
        if isinstance(image, torch.ByteTensor):
            image = image.float().div(255)
        return image, target

class Resize(BaseTransform):
    """Đổi kích thước ảnh và TỰ ĐỘNG dời tọa độ Bounding Box theo tỷ lệ tương ứng."""
    def __init__(self, size=(640, 640)):
        self.size = size

    def __call__(self, image, target):
        h, w = image.shape[:2]
        image = cv2.resize(image, self.size, interpolation=cv2.INTER_LINEAR)
        if target is not None and 'annotations' in target:
            scale_x = self.size[0] / w
            scale_y = self.size[1] / h
            for ann in target['annotations']:
                if 'bbox' in ann:
                    bbox = ann['bbox'] # [x_min, y_min, width, height]
                    ann['bbox'] = [
                        bbox[0] * scale_x,
                        bbox[1] * scale_y,
                        bbox[2] * scale_x,
                        bbox[3] * scale_y
                    ]
        return image, target

class Normalize(BaseTransform):
    """Chuẩn hóa phân phối ảnh bằng Mean và Std của ImageNet giúp hội tụ nhanh hơn."""
    def __init__(self, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
        self.mean = torch.tensor(mean).view(3, 1, 1)
        self.std = torch.tensor(std).view(3, 1, 1)

    def __call__(self, image, target):
        image = (image - self.mean) / self.std
        return image, target

# =====================================================================
# 2. ADVANCED DATA AUGMENTATION (TĂNG CƯỜNG DỮ LIỆU ĐỘNG)
# Dùng để phá vỡ ngụy trang của rác, xoay lật đa góc độ, trộn màu
# =====================================================================

class RandomHorizontalFlip(BaseTransform):
    """Lật ảnh ngang ngẫu nhiên, VÀ LẬT LẠI TỌA ĐỘ BOUNDING BOX TƯƠNG ỨNG."""
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, image, target):
        if random.random() < self.p:
            image = cv2.flip(image, 1) # 1: lật ngang
            h, w = image.shape[:2]
            if target is not None and 'annotations' in target:
                for ann in target['annotations']:
                    if 'bbox' in ann:
                        x_min, y_min, width, height = ann['bbox']
                        # Toán học lật tọa độ x_min từ bên phải sang bên trái
                        new_x_min = w - x_min - width
                        ann['bbox'] = [new_x_min, y_min, width, height]
        return image, target

class RandomColorJitter(BaseTransform):
    """Thay đổi độ sáng, độ bão hòa màu để bãi cỏ không bị học vẹt thành màu xanh."""
    def __init__(self, brightness=0.2, saturation=0.2):
        self.brightness = brightness
        self.saturation = saturation

    def __call__(self, image, target):
        # Chuyển đổi không gian màu RGB -> HSV để thay đổi tông màu
        img_hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)
        
        # Ngẫu nhiên chỉnh Saturation (Mức độ rực rỡ của màu)
        s_ratio = random.uniform(1 - self.saturation, 1 + self.saturation)
        img_hsv[:, :, 1] *= s_ratio
        
        # Ngẫu nhiên chỉnh Value (Độ sáng tối)
        v_ratio = random.uniform(1 - self.brightness, 1 + self.brightness)
        img_hsv[:, :, 2] *= v_ratio
        
        img_hsv = np.clip(img_hsv, 0, 255).astype(np.uint8)
        image = cv2.cvtColor(img_hsv, cv2.HSV2BGR)
        return image, target


# Chạy thử

In [ ]:
# TIỀN XỬ LÝ TĨNH: Xử lý raw anns -> processed ans (7 loại) -> train/test
config = AppConfig()

# Kiểm tra đường dẫn log thực tế
print(f"Đường dẫn log dự kiến: {config.system.log_file_path}")

# Khởi tạo Logger với đường dẫn mới nhất từ config
LoggerSetup.initialize(config.system.log_file_path, clear_old_logs=True)

try:
    data_pipeline = WastePreprocessingPipeline(config=config.data)
    data_pipeline.execute()
    print("\n✅ Đã hoàn thành! Hãy kiểm tra folder trên Google Drive của bạn.")
except Exception as e:
    logging.exception(f"Pipeline sập do lỗi: {e}")
    raise e

Đường dẫn log dự kiến: /content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/logs/pipeline.log
2026-06-07 15:12:48,605 - [root] - INFO - Hệ thống Logger đã sẵn sàng. File log: /content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/logs/pipeline.log
2026-06-07 15:12:48,608 - [TACO_Pipeline] - INFO - WastePreprocessingPipeline: Dây chuyền đã được cấu hình và sẵn sàng kích hoạt.
2026-06-07 15:12:48,611 - [TACO_Pipeline] - INFO - --- GIAI ĐOẠN TIỀN XỬ LÝ TĨNH (XỬ LÝ RAW ANNS) ---
2026-06-07 15:12:48,613 - [TACO_Pipeline] - INFO - [Pipeline-Step 1/4] Đang nạp tài nguyên từ ổ cứng...
2026-06-07 15:12:48,615 - [TACO_Pipeline] - INFO - Đang nạp dữ liệu từ file: /content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/data/raw/annotations.json...
2026-06-07 15:12:48,755 - [TACO_Pipeline] - INFO - Nạp thành công /content/drive/MyDrive/HCMUS/3/DL/Báo cáo cuối kỳ/Demo/data/raw/annotations.json! Dung lượng file: 2.88 MB
2026-06-07 15:12:48,757 - [TACO_Pipeline] - INFO - Đang nạp 

## Cấu hình Huấn luyện YOLOv8 đáp ứng Tiêu chí Đồ án (Rubric)
Để đảm bảo mô hình hội tụ tốt và đạt điểm tối đa cho các tiêu chí kỹ thuật trong báo cáo đồ án, quá trình huấn luyện YOLOv8 được cấu hình 3 kỹ thuật cốt lõi sau:
1. **Early Stopping (`patience=25`)**: Tự động dừng huấn luyện nếu độ đo mAP không cải thiện sau 25 epochs. Kỹ thuật này giúp tiết kiệm tài nguyên tính toán và ngăn chặn mô hình bị Học vẹt (Overfitting).
2. **Checkpointing (`save_period=10`)**: Tự động lưu trọng số dự phòng (checkpoint) mỗi 10 vòng lặp. Giúp quá trình huấn luyện có thể phục hồi nếu gặp sự cố sập nguồn.
3. **Learning Rate Scheduler (`cos_lr=True`)**: Tích hợp Cosine Annealing Scheduler. Tốc độ học (Learning Rate) sẽ giảm dần theo hình sin, giúp mô hình hội tụ từ từ và chính xác vào điểm tối ưu toàn cục ở các vòng lặp cuối.

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình YOLOv8 Medium (Tối ưu cho nhận diện vật thể nhỏ)
model = YOLO('yolov8m.pt')

# Bắt đầu huấn luyện với đầy đủ chuẩn mực của một kỹ sư AI
results = model.train(
    data='/kaggle/working/Waste-Detection-and-Classification/TACO-1/data.yaml',
    epochs=150,
    batch=16,
    imgsz=640,
    project='runs/detect',
    name='yolov8m_trashnet',
    
    # 1. Early Stopping
    patience=25,       
    
    # 2. Checkpointing 
    save=True,         
    save_period=10,    
    
    # 3. Learning Rate Scheduler 
    cos_lr=True,       
    lr0=0.01,          
    lrf=0.01,          
    
    # 4. Data Augmentation (Đã tắt các hiệu ứng bóp méo để bảo toàn rác nhỏ)
    mosaic=1.0,  
    degrees=10.0, 
    scale=0.0,    
    perspective=0.0, 
    mixup=0.0,    
    flipud=0.0,   
    fliplr=0.5
)

## Cấu hình Huấn luyện YOLOv8 đáp ứng Tiêu chí Đồ án (Rubric)
Để đảm bảo mô hình hội tụ tốt và đạt điểm tối đa cho các tiêu chí kỹ thuật trong báo cáo đồ án, quá trình huấn luyện YOLOv8 được cấu hình 3 kỹ thuật cốt lõi sau:
1. **Early Stopping (`patience=25`)**: Tự động dừng huấn luyện nếu độ đo mAP không cải thiện sau 25 epochs. Kỹ thuật này giúp tiết kiệm tài nguyên tính toán và ngăn chặn mô hình bị Học vẹt (Overfitting).
2. **Checkpointing (`save_period=10`)**: Tự động lưu trọng số dự phòng (checkpoint) mỗi 10 vòng lặp. Giúp quá trình huấn luyện có thể phục hồi nếu gặp sự cố sập nguồn.
3. **Learning Rate Scheduler (`cos_lr=True`)**: Tích hợp Cosine Annealing Scheduler. Tốc độ học (Learning Rate) sẽ giảm dần theo hình sin, giúp mô hình hội tụ từ từ và chính xác vào điểm tối ưu toàn cục ở các vòng lặp cuối.

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình YOLOv8 Medium (Tối ưu cho nhận diện vật thể nhỏ)
model = YOLO('yolov8m.pt')

# Bắt đầu huấn luyện với đầy đủ chuẩn mực của một kỹ sư AI
results = model.train(
    data='/kaggle/working/Waste-Detection-and-Classification/TACO-1/data.yaml',
    epochs=150,
    batch=16,
    imgsz=640,
    project='runs/detect',
    name='yolov8m_trashnet',
    
    # 1. Early Stopping
    patience=25,       
    
    # 2. Checkpointing 
    save=True,         
    save_period=10,    
    
    # 3. Learning Rate Scheduler 
    cos_lr=True,       
    lr0=0.01,          
    lrf=0.01,          
    
    # 4. Data Augmentation (Đã tắt các hiệu ứng bóp méo để bảo toàn rác nhỏ)
    mosaic=1.0,  
    degrees=10.0, 
    scale=0.0,    
    perspective=0.0, 
    mixup=0.0,    
    flipud=0.0,   
    fliplr=0.5
)